# SignSparK zero-shot check: English text → 3D signing

Before spending weeks converting Auslan-Daily to SMPL-X, this notebook checks whether the released SignSparK checkpoints (ECCV 2026; trained on CSL-Daily, How2Sign, BOBSL, CSL-News) produce **moving, articulate signing from text alone**. It needs no Auslan data.

Three sampling runs, each over the three streams (hand, body, face):

| run | clips | conditioning | what it tells you |
|---|---|---|---|
| A `h2s_kf` | 8 How2Sign test clips | text + ground-truth keyframes (the official setting) | the install works; upper bound |
| B `h2s_text` | the same 8 clips | text only, classifier-free guidance, all keyframes masked | the real use case, with ground truth to compare against |
| C `demo_text` | your everyday sentences | text only, as B | what it does on new English input |

Facts checked in the code/paper that shape this notebook:
- `sample.py` only samples from an LMDB, so run C writes a small LMDB with the sentences, placeholder (identity) poses and **a length we choose** — the paper does not say how length is set at test time. Length comes from a words→frames fit on How2Sign test.
- Text-only generation is what the paper does for text-to-pose (keyframes masked, CFG); the model saw keyframe-free inputs in 10% of training.
- The LMDB `language` field is the **spoken** language (e.g. `German`), so How2Sign (ASL) and BOBSL (BSL) are probably both `English`: a BSL-specific tag is not guaranteed to exist. Run C samples every sentence with both `<English>` and `<BSL>` so you can compare.
- Output is SMPL-X rotations (not our 2D keypoints). Section 7 needs no extra files; sections 8–9 need the **SMPL-X model** (license-gated): register at https://smpl-x.is.tue.mpg.de/, download SMPL-X v1.1, and put `SMPLX_NEUTRAL.npz` on Drive at `MyDrive/smplx_models/smplx/SMPLX_NEUTRAL.npz`.

License: code Apache-2.0; checkpoints non-commercial research only. Needs ~20 GB of local disk (three 5.6 GB checkpoints) and an A100.

## 1. Runtime, code and dependencies

In [1]:
import os, sys, subprocess, glob, json, io, pickle, shutil, re
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() or 'NO GPU')

SSK = '/content/SignSparK'
SSK_COMMIT = '7b9b48360c4f5c26ceb0ff5a8d88424ade4e586d'   # main on 2026-09-17
if not os.path.isdir(f'{SSK}/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/JianHe0628/SignSparK.git', SSK], check=True)
subprocess.run(['git', '-C', SSK, 'checkout', '-q', SSK_COMMIT], check=True)

# requirements.txt minus torch/torchvision (Colab's CUDA build is kept) and the pyrender stack
# (section 9 draws its own skeleton, so no EGL setup is needed).
!pip -q install torchdiffeq==0.2.5 accelerate==1.10.1 transformers==4.56.1 "huggingface_hub<1.0" sentence-transformers==5.1.2 multilingual-clip==1.0.10 hydra-core==1.3.2 omegaconf==2.3.0 einops==0.8.1 lmdb==2.2.0 blobfile==3.0.0 wandb==0.21.3 smplx
print(subprocess.run([sys.executable, '-c', 'import torch, transformers, numpy; print("torch", torch.__version__, "| transformers", transformers.__version__, "| numpy", numpy.__version__, "| cuda", torch.cuda.is_available())'], capture_output=True, text=True).stdout)

NVIDIA A100-SXM4-40GB, 40960 MiB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 473.2 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 2.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 40.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.0/488.0 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.0/336.0 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 41.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.8/22

## 2. Drive (outputs) and the SMPL-X model

In [2]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
OUT = f'{DRIVE}/signspark_zeroshot'
os.makedirs(f'{OUT}/samples', exist_ok=True)

hits = sorted(glob.glob(f'{DRIVE}/smplx_models/smplx/SMPLX_NEUTRAL.npz') + glob.glob(f'{DRIVE}/**/smplx/SMPLX_NEUTRAL.npz', recursive=True))
SMPLX_DIR = os.path.dirname(os.path.dirname(hits[0])) if hits else None
print('SMPL-X model folder:', SMPLX_DIR or 'NOT FOUND - sections 8-9 will be skipped (see the note at the top)')

Mounted at /content/drive
SMPL-X model folder: /content/drive/MyDrive/smplx_models


## 3. Download checkpoints and the How2Sign test LMDB (pinned revisions)

Checkpoints go to local disk (`/content/ssk_ckpt`, ~17 GB); only samples and videos are written to Drive.

In [3]:
from huggingface_hub import snapshot_download
MODEL_REV = 'b907f25f9bb9f5a0adcaf131a23a155dc471d9a8'
DATA_REV = '9fe7e04ef59204c787cd6094c747f889efff6ba0'
CKPT = '/content/ssk_ckpt'
DATA = '/content/ssk_data'
snapshot_download('LionelLow/SignSparK', revision=MODEL_REV, local_dir=CKPT, allow_patterns=['hand/*', 'body/*', 'face/*'])
snapshot_download('LionelLow/SignSparK_data', repo_type='dataset', revision=DATA_REV, local_dir=f'{DATA}/lmdb', allow_patterns=['test/How2Sign*'])
STREAMS = ['hand', 'body', 'face']
for s in STREAMS:
    p = f'{CKPT}/{s}/ema_0.9999_200000.pt'
    print(f'{s}: {os.path.getsize(p)/1e9:.2f} GB')
H2S = f'{DATA}/lmdb/test/How2Sign_reopt_test.lmdb'
print('How2Sign test LMDB:', os.path.getsize(f'{H2S}/data.mdb') / 1e6, 'MB')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

hand/ema_0.9999_200000.pt:   0%|          | 0.00/5.58G [00:00<?, ?B/s]

body/ema_0.9999_200000.pt:   0%|          | 0.00/5.58G [00:00<?, ?B/s]

face/ema_0.9999_200000.pt:   0%|          | 0.00/5.58G [00:00<?, ?B/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

test/How2Sign_reopt_test.lmdb/data.mdb:   0%|          | 0.00/588M [00:00<?, ?B/s]

hand: 5.58 GB
body: 5.58 GB
face: 5.58 GB
How2Sign test LMDB: 587.984896 MB


## 4. How2Sign test: language tag, lengths, and the length model for new sentences

The first `N_H2S` clips (loader order) are the ones runs A and B sample.

In [4]:
import numpy as np, lmdb

def read_lmdb(path, limit=None):
    env = lmdb.open(path, readonly=True, lock=False, readahead=False)
    with env.begin() as txn:
        ids = pickle.loads(txn.get(b'__meta__'))['clip_ids']
        out = []
        for k in ids[:limit]:
            z = np.load(io.BytesIO(txn.get(k.encode())), allow_pickle=True)
            out.append({'id': k, 'language': str(z['language'][0]), 'text': str(z['translation'][0]), 'T': len(z['segment'])})
    env.close()
    return out

h2s = read_lmdb(H2S)
T = np.array([r['T'] for r in h2s]); W = np.array([len(r['text'].split()) for r in h2s])
print(f'{len(h2s)} clips | language tags {sorted({r["language"] for r in h2s})}')
print(f'frames: median {np.median(T):.0f}, 5-95% {np.percentile(T, 5):.0f}-{np.percentile(T, 95):.0f} | words median {np.median(W):.0f}')
b, a = np.polyfit(W, T, 1)
print(f'length model: frames ≈ {a:.1f} + {b:.2f} × words (fit on How2Sign test, capped to the 304-frame window)')
def est_len(text):
    return int(np.clip(round(a + b * len(text.split())), 32, 300))

N_H2S = 8
print('\nclips used by runs A/B:')
for r in h2s[:N_H2S]:
    print(f"  {r['T']:>4} frames | {r['text'][:90]}")

2183 clips | language tags ['American']
frames: median 140, 5-95% 37-477 | words median 15
length model: frames ≈ 12.6 + 9.26 × words (fit on How2Sign test, capped to the 304-frame window)

clips used by runs A/B:
   651 frames | The aileron is the control surface in the wing that is controlled by lateral movement righ
   214 frames | By moving the stick, you cause pressure to increase or decrease the angle of attack on tha
   844 frames | The elevator is the part that moves with the stick forward and back, and that adjusts the 
   288 frames | Therefore, it's either going uphill, downhill, or flat and that adjusts the air speed that
   547 frames | The rudder is the vertical stabilizer.
   227 frames | That's moved by the feet and that's what actually steers the airplane right and left when 
   278 frames | Buenos Dias, I'm Bobby Larew, you didn't know I spoke Spanish, did you?
   294 frames | I'm an expert on diving, talking about a back 1 1/2 pike.


## 5. Write the everyday sentences as an LMDB (run C)

Each record follows DATA.md: text + language tag, an all-zero `segment` (no sign segments → no keyframes), and identity-rotation placeholder poses (zeros would turn into NaN in the left-hand flip). Edit `SENTENCES` / `TAGS` freely.

In [5]:
SENTENCES = [
    'hello, how are you?',
    'what is your name?',
    'i am hungry, let us eat.',
    'thank you very much.',
    'see you tomorrow.',
    'my family lives in melbourne.',
]
TAGS = ['English', 'BSL']
DEMO_NAME = 'AuslanDemo'
DEMO = f'{DATA}/lmdb/test/{DEMO_NAME}_test.lmdb'

ID6 = np.array([1, 0, 0, 0, 1, 0], np.float32)
def placeholder(T, joints, extra=0):
    x = np.tile(ID6, (T, joints))
    return np.concatenate([x, np.zeros((T, extra), np.float32)], 1) if extra else x

def write_demo_lmdb(path, sentences, tags):
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    env = lmdb.open(path, map_size=1 << 30)
    ids, rows = [], []
    with env.begin(write=True) as txn:
        for tag in tags:
            for i, s in enumerate(sentences):
                n = est_len(s)
                cid = f'{tag}_{i:02d}'
                buf = io.BytesIO()
                np.savez(buf, language=np.array([tag], dtype=object), translation=np.array([s], dtype=object),
                         gloss=np.array([''], dtype=object), segment=np.zeros(n, np.int32),
                         left_features=placeholder(n, 16), right_features=placeholder(n, 16),
                         body_features=placeholder(n, 21), face_features=placeholder(n, 1, extra=50))
                txn.put(cid.encode(), buf.getvalue())
                ids.append(cid); rows.append((cid, n, s))
        txn.put(b'__meta__', pickle.dumps({'clip_ids': ids, 'num_clips': len(ids)}))
    env.close()
    return rows

demo_rows = write_demo_lmdb(DEMO, SENTENCES, TAGS)
for cid, n, s in demo_rows:
    print(f'{cid:<12} {n:>4} frames | {s}')
N_DEMO = len(demo_rows)

English_00     50 frames | hello, how are you?
English_01     50 frames | what is your name?
English_02     68 frames | i am hungry, let us eat.
English_03     50 frames | thank you very much.
English_04     40 frames | see you tomorrow.
English_05     59 frames | my family lives in melbourne.
BSL_00         50 frames | hello, how are you?
BSL_01         50 frames | what is your name?
BSL_02         68 frames | i am hungry, let us eat.
BSL_03         50 frames | thank you very much.
BSL_04         40 frames | see you tomorrow.
BSL_05         59 frames | my family lives in melbourne.


## 6. Sample

Each call runs the repo's own `sample.py` for one stream. Runs B and C use classifier-free guidance with all keyframes masked; `TEXT_SCALE` is not given in the paper, so 2.5 is a starting point (try 1.5 / 4 if the motion looks too flat or too wild). Finished samples are copied to Drive and skipped on re-run.

With guidance on, `sample.py` drops incomplete batches, so the batch size is set to exactly the number of clips.

In [14]:
os.makedirs(f'{OUT}/samples', exist_ok=True)
os.makedirs(f'{OUT}/videos', exist_ok=True)


In [15]:
ODE_STEPS = 20        # README: fewer than ten steps already works; sample_all.py uses 20
TEXT_SCALE = 1.5

def sample(stream, dataset, n, note, text_only):
    dst = f'{OUT}/samples/{note}_{stream}.npy'
    if os.path.exists(dst):
        return dst
    args = [sys.executable, 'sample.py', f'data_v2={stream}', 'data_v2/common@common=inference',
            f'eval.model_path={CKPT}/{stream}/ema_0.9999_200000.pt', f'eval.note={note}',
            f'eval.ode_stepnum={ODE_STEPS}', 'eval.split=test', f'test_data=[{dataset}]',
            f'eval.batch_size={n}', f'eval.max_samples={n}', 'flip_left_hand=True', f'base_path={DATA}/lmdb',
            f'eval.text_guidance_scale={TEXT_SCALE if text_only else -1}', f'eval.keyframes_mask_all={text_only}']
    env = dict(os.environ, WANDB_MODE='disabled', OUTPUT_DIR='/content/ssk_runs', DATA_ROOT=DATA, SIGNSPARK_CKPT_DIR=CKPT)
    res = subprocess.run(args, cwd=SSK, env=env, capture_output=True, text=True)
    log = res.stdout + res.stderr
    open(f'{OUT}/samples/{note}_{stream}.log', 'w').write(log)
    m = re.search(r'### Written the decoded output to (\S+\.npy)', log)
    if res.returncode != 0 or not m:
        print(log[-4000:])
        raise RuntimeError(f'sample.py failed: {note} / {stream}')
    shutil.copyfile(m.group(1), dst)
    return dst

RUNS = {  # note: (dataset, n clips, text only)
    'h2s_kf':    ('How2Sign', N_H2S, False),
    'h2s_text':  ('How2Sign', N_H2S, True),
    'demo_text': (DEMO_NAME, N_DEMO, True),
}
SAMPLES = {}
for note, (dataset, n, text_only) in RUNS.items():
    for s in STREAMS:
        SAMPLES[note, s] = sample(s, dataset, n, note, text_only)
        print(f'{note:<10} {s:<5} -> {SAMPLES[note, s]}')

h2s_kf     hand  -> /content/drive/MyDrive/signspark_zeroshot/samples/h2s_kf_hand.npy
h2s_kf     body  -> /content/drive/MyDrive/signspark_zeroshot/samples/h2s_kf_body.npy
h2s_kf     face  -> /content/drive/MyDrive/signspark_zeroshot/samples/h2s_kf_face.npy
h2s_text   hand  -> /content/drive/MyDrive/signspark_zeroshot/samples/h2s_text_hand.npy
h2s_text   body  -> /content/drive/MyDrive/signspark_zeroshot/samples/h2s_text_body.npy
h2s_text   face  -> /content/drive/MyDrive/signspark_zeroshot/samples/h2s_text_face.npy
demo_text  hand  -> /content/drive/MyDrive/signspark_zeroshot/samples/demo_text_hand.npy
demo_text  body  -> /content/drive/MyDrive/signspark_zeroshot/samples/demo_text_body.npy
demo_text  face  -> /content/drive/MyDrive/signspark_zeroshot/samples/demo_text_face.npy


## 7. Does it move? (no SMPL-X needed)

Mean frame-to-frame change of the 6D rotation features inside each clip's length, per stream. For How2Sign the ground truth of the same clips is the reference; for the demo sentences the How2Sign ground truth is the reference. A ratio near 1 means realistic amounts of motion; our 2D model scored 0.01 (frozen). This does **not** say the signs are right — only that the model is not collapsing to a still pose.

In [16]:
def flat(x):
    return x.reshape(x.shape[0], -1, x.shape[-1])          # (B, C, T), as tools/visualize.py::load_stream

def load(note, stream):
    d = np.load(SAMPLES[note, stream], allow_pickle=True).item()
    return flat(d['pred_poses'][0]), flat(d['gt_poses'][0]), d['lengths'][0], list(d['text'][0])   # hand: B = 2 x clips (L, R interleaved)

def motion(x, lengths):
    return float(np.mean([np.abs(np.diff(x[i, :, :int(L)], axis=-1)).mean() for i, L in enumerate(lengths) if L > 1]))

print(f"{'stream':<6}{'GT (How2Sign)':>15}{'A keyframes':>13}{'B text-only':>13}{'C demo':>9}   ratios A / B / C vs GT")
for s in STREAMS:
    pa, gt, la, _ = load('h2s_kf', s)
    pb, _, lb, _ = load('h2s_text', s)
    pc, _, lc, _ = load('demo_text', s)
    g, ma, mb, mc = motion(gt, la), motion(pa, la), motion(pb, lb), motion(pc, lc)
    print(f'{s:<6}{g:>15.4f}{ma:>13.4f}{mb:>13.4f}{mc:>9.4f}   {ma/g:.2f} / {mb/g:.2f} / {mc/g:.2f}')

# Keyframe-free sampling is stochastic: a quick check that different sentences give different motion.
pc, _, lc, texts = load('demo_text', 'hand')
L = int(min(lc))
vec = pc[:, :, :L].reshape(len(pc), -1)
dist = np.linalg.norm(vec[:, None] - vec[None], axis=-1)
print(f'\nhand stream, demo clips: mean pairwise distance {dist[np.triu_indices(len(pc), 1)].mean():.2f} '
      f'(0 would mean every sentence produced the same signing)')

stream  GT (How2Sign)  A keyframes  B text-only   C demo   ratios A / B / C vs GT
hand           0.0132       0.0144       0.0203   0.0199   1.09 / 1.53 / 1.50
body           0.0068       0.0072       0.0089   0.0136   1.06 / 1.31 / 2.01
face           0.0538       0.0580       0.0759   0.1017   1.08 / 1.41 / 1.89

hand stream, demo clips: mean pairwise distance 15.53 (0 would mean every sentence produced the same signing)


## 8. SMPL-X forward kinematics

Uses the repo's own `tools/visualize.py::build_smplx_input` (6D → axis-angle, left-hand flip back to SMPL-X convention, neutral legs), then SMPL-X joints. The released face stream is rendered with SMPL-X jaw + expression, as in the repo.

In [17]:
def flat(x):
    return x.reshape(x.shape[0], -1, x.shape[-1])


In [18]:
import torch
sys.path.insert(0, SSK)
from tools.visualize import build_smplx_input

JOINTS = {}
if SMPLX_DIR is None:
    print('SMPL-X model not found; skipping')
else:
    import smplx
    dev = 'cuda'
    SMPLX = smplx.create(SMPLX_DIR, model_type='smplx', gender='neutral', use_pca=False, flat_hand_mean=True,
                         num_betas=10, num_expression_coeffs=50).to(dev).eval()
    PARENTS = SMPLX.parents.cpu().numpy()

    def fk(body, lh, rh, face):
        kw = build_smplx_input(torch.from_numpy(body).float(), torch.from_numpy(lh[:, :90]).float(),
                               torch.from_numpy(rh[:, :90]).float(), face=torch.from_numpy(face).float(), device=dev)
        with torch.no_grad():
            return SMPLX(**kw).joints.cpu().numpy()        # (T, 127, 3)

    for note in RUNS:
        streams = {s: np.load(SAMPLES[note, s], allow_pickle=True).item() for s in STREAMS}
        lengths = streams['body']['lengths'][0]
        texts = list(streams['body']['text'][0])
        names = [str(n) for n in streams['body']['video_names'][0]]
        out = []
        for key in ('pred_poses', 'gt_poses') if note.startswith('h2s') else ('pred_poses',):
            b, h, f = (flat(streams[s][key][0]).transpose(0, 2, 1) for s in ('body', 'hand', 'face'))   # (B, T, C)
            clips = []
            for i, n in enumerate(lengths):
                n = int(n)
                clips.append(fk(b[i, :n], h[2 * i, :n], h[2 * i + 1, :n], f[i, :n]))
            out.append(clips)
        JOINTS[note] = {'pred': out[0], 'gt': out[1] if len(out) > 1 else None, 'text': texts, 'names': names}
        print(f'{note}: {len(out[0])} clips')

h2s_kf: 8 clips
h2s_text: 8 clips
demo_text: 12 clips


## 9. Render skeleton videos

Front view, upper body, both hands with fingertips, and the SMPL-X face landmarks (so jaw/mouth movement is visible). How2Sign runs show ground truth (left) next to the generation (right). `FPS` is only playback speed; the dataset frame rate is not stated in the repo.

In [19]:
import cv2
from IPython.display import Video, display

FPS = 25
LEGS = {1, 2, 4, 5, 7, 8, 10, 11}
FEET = set(range(60, 66))
LEFT = {16, 18, 20} | set(range(25, 40)) | set(range(66, 71))
RIGHT = {17, 19, 21} | set(range(40, 55)) | set(range(71, 76))
TIPS = [(39, 66), (27, 67), (30, 68), (36, 69), (33, 70), (54, 71), (42, 72), (45, 73), (51, 74), (48, 75)]
FACE_LMK = range(76, 127)

def bones(parents):
    return [(j, int(parents[j])) for j in range(1, 55) if j not in LEGS and int(parents[j]) not in LEGS and parents[j] >= 0] + TIPS

def draw(J, size, box, bone_list, title):
    img = np.full((size, size, 3), 255, np.uint8)
    (x0, y0), s = box
    P = np.stack([(J[:, 0] - x0) * s + size / 2, (y0 - J[:, 1]) * s + size / 2], 1).astype(int)
    for a, b in bone_list:
        colour = (0, 90, 230) if a in RIGHT else (230, 120, 0) if a in LEFT else (60, 60, 60)   # BGR
        cv2.line(img, tuple(P[a]), tuple(P[b]), colour, 2, cv2.LINE_AA)
    for k in FACE_LMK:
        cv2.circle(img, tuple(P[k]), 1, (40, 40, 40), -1, cv2.LINE_AA)
    cv2.putText(img, title, (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
    return img

def view_box(clips, size):
    J = np.concatenate([c[:, [j for j in range(127) if j not in LEGS | FEET | {0}]] for c in clips if c is not None])
    lo, hi = J[..., :2].reshape(-1, 2).min(0), J[..., :2].reshape(-1, 2).max(0)
    return ((lo + hi) / 2), 0.9 * size / max(hi - lo)

def write_video(path, panels, texts, size=480):
    box = view_box(panels, size)
    bl = bones(PARENTS)
    T = max(len(p) for p in panels)
    tmp = path + '.mp4v.mp4'
    vw = cv2.VideoWriter(tmp, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (size * len(panels), size))
    for t in range(T):
        frame = np.concatenate([draw(p[min(t, len(p) - 1)], size, box, bl, lab) for p, lab in zip(panels, texts)], 1)
        vw.write(frame)
    vw.release()
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', tmp, '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', path], check=True)
    os.remove(tmp)

if JOINTS:
    os.makedirs(f'{OUT}/videos', exist_ok=True)
    for note, d in JOINTS.items():
        for i, pred in enumerate(d['pred']):
            path = f"{OUT}/videos/{note}_{i:02d}.mp4"
            if d['gt'] is not None:
                write_video(path, [d['gt'][i], pred], ['ground truth', 'generated'])
            else:
                write_video(path, [pred], [d['names'][i]])
    print('videos in', f'{OUT}/videos')

videos in /content/drive/MyDrive/signspark_zeroshot/videos


Show the videos: How2Sign first (is text-only B close to A and to the ground truth?), then the everyday sentences (`English_*` vs `BSL_*` tags).

In [20]:
SHOW = {'h2s_kf': 3, 'h2s_text': 3, 'demo_text': 99}
if JOINTS:
    for note, n in SHOW.items():
        d = JOINTS[note]
        for i in range(min(n, len(d['pred']))):
            print(f"{note} {i:02d} | {d['names'][i]} | {d['text'][i]}")
            display(Video(f'{OUT}/videos/{note}_{i:02d}.mp4', embed=True, width=480 * (2 if d['gt'] is not None else 1)))

h2s_kf 00 | -fZc293MpJk_2-1-rgb_front | <American> The aileron is the control surface in the wing that is controlled by lateral movement right and left of the stick.


h2s_kf 01 | -fZc293MpJk_3-1-rgb_front | <American> By moving the stick, you cause pressure to increase or decrease the angle of attack on that particular raising or lowering the wing.


h2s_kf 02 | -fZc293MpJk_4-1-rgb_front | <American> The elevator is the part that moves with the stick forward and back, and that adjusts the angle of attack of the airplane in the air.


h2s_text 00 | -fZc293MpJk_2-1-rgb_front | <American> The aileron is the control surface in the wing that is controlled by lateral movement right and left of the stick.


h2s_text 01 | -fZc293MpJk_3-1-rgb_front | <American> By moving the stick, you cause pressure to increase or decrease the angle of attack on that particular raising or lowering the wing.


...

demo_text 07 | BSL_01 | <BSL> what is your name?


demo_text 08 | BSL_02 | <BSL> i am hungry, let us eat.


demo_text 09 | BSL_03 | <BSL> thank you very much.


demo_text 10 | BSL_04 | <BSL> see you tomorrow.


demo_text 11 | BSL_05 | <BSL> my family lives in melbourne.
